# DETAILED TUTORIAL: USING DATA SUITE ON SYNTHETIC DATA

## Imports

In [1]:
import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score

import wandb
from sklearn import linear_model
from sklearn.metrics import mean_squared_error

from data_suite.data.data_loader import load_adult_data
from data_suite.utils.data_utils import (
    covariance_comparison,
    get_suspect_features,
    read_from_file,
    write_to_file,
)
from data_suite.utils.helpers import inlier_outlier_dicts, sort_ci_vals
from data_suite.utils.data_utils import (
    covariance_comparison,
    get_suspect_features,
)


model_ids = {}
artifact_path = "artifacts"



In [2]:
import logging
logger = logging.getLogger()
logger.setLevel(logging.INFO)

## Generate synthetic data

In [3]:
props = [0.1, 0.25, 0.5, 0.75]
dists = ["normal", "beta", "gamma", "weibull"]
noise_vars = [1, 2, 5, 10]
copula_count_samples = [1000]

prop = props[0]
dist = dists[0]
noise_variance = noise_vars[0]
copula_n_samples = 1000


n_synthetic = 1000
train_prop = 1
rep_type = "pca"

wandb_dict = {}


from data_suite.data.data_loader import load_synthetic_data

(
    train,
    test,
    orig_test,
    noise_bool,
    noise_matrix,
    noise_idx,
) = load_synthetic_data(
    n_synthetic=n_synthetic,
    mean=0,
    noise_variance=noise_variance,
    dim="small",
    prop=prop,
    dist=dist,
)


suspect_features = list(range(train.shape[1]))




## STEP 1: COPULA - We fit and sample a copula on the dataset. This step is optional, but it allows the user to only need synthetic data rather than real data

In [4]:
from data_suite.models.copula import fit_sample_copula

copula_samples = fit_sample_copula(
    clean_corpus=train,
    copula="vine",
    copula_n_samples=copula_n_samples,
)

INFO:root:Vine...
/home/rob/miniconda3/envs/ds_5/lib/python3.9/site-packages/copulas/multivariate/vine.py:68: UserWarning: Vines have not been fully tested on Python 3.8 and might produce wrong results. Please use Python 3.5, 3.6 or 3.7
  warnings.warn(
INFO:copulas.multivariate.vine:Fitting VineCopula("direct")


INFO:root:Copula Samples = 1000


## STEP 2: REPRESENTER - learns a low dimensional representation of the data. The representation dimension is half, but can be adjusted as a hyperparameter

In [5]:
from data_suite.models.representation import compute_representation

rep_dim = int(np.ceil(train.shape[1] / 2))
pcs_train, pcs_test, pcs_copula = compute_representation(
    train,
    test,
    copula_samples,
    n_components=rep_dim,
    rep_type=rep_type,
)


2025-01-21 16:51:22.312772: I tensorflow/core/platform/cpu_feature_guard.cc:193] This TensorFlow binary is optimized with oneAPI Deep Neural Network Library (oneDNN) to use the following CPU instructions in performance-critical operations:  AVX2 AVX_VNNI FMA
To enable them in other operations, rebuild TensorFlow with the appropriate compiler flags.
2025-01-21 16:51:22.393934: I tensorflow/core/util/port.cc:104] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2025-01-21 16:51:22.396301: W tensorflow/compiler/xla/stream_executor/platform/default/dso_loader.cc:64] Could not load dynamic library 'libcudart.so.11.0'; dlerror: libcudart.so.11.0: cannot open shared object file: No such file or directory; LD_LIBRARY_PATH: /usr/lib/gcc/x86_64-linux-gnu/11/:/usr/lib/gcc/x86_64-linux-gnu/11/:
2025-01-21 16:51:22.39631

# STEP 3: CONFORMAL PREDICTOR - a feature-wise conformal predictor is fit and each reconstruction assessed

In [6]:
from data_suite.models.conformal import conformal_class

conformal_dict = {}
for feat in suspect_features:
    feat = int(feat)
    dim = pcs_copula.shape[1]
    conf = conformal_class(
        conformity_score="sign", input_dim=dim
    )
    conf.fit(
        x_train=pcs_copula, y_train=copula_samples[:, feat]
    )
    conformal_dict[feat] = conf.predict(
        x_test=pcs_test, y_test=test[:, feat]
    )
    logging.info(f"Running analysis for feature = {feat}")


INFO:root:Running analysis for feature = 0
INFO:root:Running analysis for feature = 1
INFO:root:Running analysis for feature = 2


## PROCESS CONFORMAL INTERVALS - we need to process the intervals 

In [7]:
from data_suite.utils.helpers import *

proportion=0.4

inliers_dict, outliers_dict = inlier_outlier_dicts(
    conformal_dict, suspect_features
)


small_ci_ids, large_ci_ids, df_sorted = sort_cis_synth(
    conformal_dict, inliers_dict, suspect_features=[0], proportion=proportion
)

/home/rob/Documents/projects/Data-SUITE/src/data_suite/utils/helpers.py:79: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_inlier[f"{feature}_contrib"] = df_inlier["norm_interval"]


## EXAMPLE: TRAIN A DOWNSTREAM REGRESSION MODEL & SHOW RESULTS ON THE DIFFERENT TYPES OF SAMPLES

In [8]:
from data_suite.models.benchmarks import comparison_methods

inlier_ids = inliers_dict[0]
outlier_ids = outliers_dict[0]


#####################################
#
# FIT A DOWNSTREAM MODEL ON TRAINING DATA
# MAKE PREDICTIONS ON TEST DATA
#
#####################################

# Create linear regression object
regr = linear_model.LinearRegression()

# Train the model using the training sets
regr.fit(train[:, 0:-1], train[:, -1])

y_pred = regr.predict(train[:, 0:-1])


#####################################
#
# ASSESS MSE ON THE DIFFERENT TYPE OF SAMPLES IDENTIFIED
#
#####################################

mse = mean_squared_error(train[:, -1], y_pred)
print(f"MSE Train data: {mse} \n")
wandb_dict["mse_train_clean"] = mse

print("-------------------------------")

y_pred = regr.predict(test[:, 0:-1])
mse = mean_squared_error(test[:, -1], y_pred)
print(
    f"MSE Test (ALL SAMPLES - INLIERS+OUTLIERS): {mse} \n"
)
wandb_dict["mse_test_unknown"] = mse

print("-------------------------------")

y_pred = regr.predict(test[outlier_ids, 0:-1])
mse = mean_squared_error(test[outlier_ids, -1], y_pred)
print(f"MSE Outliers: {mse} \n")
wandb_dict["mse_test_outliers"] = mse

print("-------------------------------")

y_pred = regr.predict(test[inlier_ids, 0:-1])
mse = mean_squared_error(test[inlier_ids, -1], y_pred)
print(f"MSE Inliers: {mse } \n")
wandb_dict["mse_test_inliers"] = mse

y_pred = regr.predict(test[small_ci_ids, 0:-1])
mse = mean_squared_error(test[small_ci_ids, -1], y_pred)
print(f"MSE Inliers w/ SMALL CIs: {mse}\n ")
wandb_dict["mse_test_inliers_small_ci"] = mse

y_pred = regr.predict(test[large_ci_ids, 0:-1])
mse = mean_squared_error(test[large_ci_ids, -1], y_pred)
print(f"MSE Inliers w/  LARGE CIs: {mse}\n")
wandb_dict["mse_test_inliers_large_ci"] = mse

MSE Train data: 0.07411232463191746 

-------------------------------
MSE Test (ALL SAMPLES - INLIERS+OUTLIERS): 0.10763254733082346 

-------------------------------
MSE Outliers: 0.28261383384341027 

-------------------------------
MSE Inliers: 0.08653972671555409 

MSE Inliers w/ SMALL CIs: 0.06718630020116077
 
MSE Inliers w/  LARGE CIs: 0.1119687109189934



### Note the differences between samples with small CIs and Large CIs - indicating we can trust samples with small CIs more

## EXAMPLE: Compute performance metrics

In [9]:
from data_suite.utils.uncertainty_metrics import *
from copy import deepcopy

ids = range(test.shape[0])

ids = inlier_ids

y_test_ids = noise_bool

x_train_uncert, y_train_uncert = train[:, 1:], train[:, 0]
x_test_uncert = test[:, 1:]
feature=0

df_conformal = conformal_dict[feature]

df_conformal = df_conformal.iloc[ids, :]

df_conformal["pred"] = df_conformal["min"] + (df_conformal["conf_interval"] / 2)

preds = df_conformal["pred"]  # target predictions
# dc['true_val']  # ground truth observations
true = orig_test[ids, 0]
# lower bound of the prediction interval
lb = df_conformal["min"]
# upper bound of the prediction interval
ub = df_conformal["max"]

print("COMPUTING PERFORMANCE METRICS")

(
    uncert_metrics,
    excess,
    deficet,
    excess_all,
    deficet_all,
) = compute_uncertainty_metrics(
    preds=preds, lower_bound=lb, upper_bound=ub, true=true
)

idx_ordered = list(df_conformal.sort_values(by="conf_interval").index)
results, roc = test_ood(np.array(y_test_ids)[ids], idx_ordered)

wandb_dict = process_results(
    wandb_dict,
    results,
    roc,
    uncert_metrics,
    excess,
    deficet,
    excess_all,
    deficet_all,
    name="conformal_copula",
)

/tmp/ipykernel_350381/1346147772.py:18: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_conformal["pred"] = df_conformal["min"] + (df_conformal["conf_interval"] / 2)


COMPUTING PERFORMANCE METRICS
KNNs ROC:0.4595, precision @ rank n:0.9545
KNNs ROC:0.4651, precision @ rank n:0.9545
KNNs ROC:0.4651, precision @ rank n:0.9545
KNNs ROC:0.4699, precision @ rank n:0.9545
KNNs ROC:0.4699, precision @ rank n:0.9545
KNNs ROC:0.4741, precision @ rank n:0.9545
KNNs ROC:0.4779, precision @ rank n:0.9545
KNNs ROC:0.4779, precision @ rank n:0.9545
KNNs ROC:0.4812, precision @ rank n:0.9545
KNNs ROC:0.4812, precision @ rank n:0.9545
KNNs ROC:0.4842, precision @ rank n:0.9545
KNNs ROC:0.4869, precision @ rank n:0.9545
KNNs ROC:0.4869, precision @ rank n:0.9545
KNNs ROC:0.4894, precision @ rank n:0.9545
KNNs ROC:0.4894, precision @ rank n:0.9545
KNNs ROC:0.4916, precision @ rank n:0.9545
KNNs ROC:0.4916, precision @ rank n:0.9545
KNNs ROC:0.4936, precision @ rank n:0.9545
KNNs ROC:0.4955, precision @ rank n:0.9545
KNNs ROC:0.4955, precision @ rank n:0.9545
KNNs ROC:0.4972, precision @ rank n:0.9545
KNNs ROC:0.4972, precision @ rank n:0.9545
KNNs ROC:0.4988, precisi

## This dict could be logged - note PICP (Prediction interval coverage probability) = Coverage 

In [10]:
wandb_dict

{'mse_train_clean': 0.07411232463191746,
 'mse_test_unknown': 0.10763254733082346,
 'mse_test_outliers': 0.28261383384341027,
 'mse_test_inliers': 0.08653972671555409,
 'mse_test_inliers_small_ci': 0.06718630020116077,
 'mse_test_inliers_large_ci': 0.1119687109189934,
 'excess_conformal_copula': 1.313843634419015,
 'deficet_conformal_copula': 0.12026163812730511,
 'excess_all_conformal_copula': 1.2803841191112302,
 'deficet_all_conformal_copula': 0.003062690274888925,
 'roc_conformal_copula': 0.4595,
 'rmse_conformal_copula': 0.48521990979766216,
 'nll_conformal_copula': 0.7055475709674863,
 'auucc_gain_conformal_copula': -0.30819231521252866,
 'picp_conformal_copula': 0.9745331069609507,
 'mpiw_conformal_copula': 3.2662268040286158,
 'r2_conformal_copula': 0.9339644027456961,
 'TPR_conformal_copula': 0.9543285616905249,
 'FPR_conformal_copula': 0.9797985358066824,
 'TNR_conformal_copula': 0.020183115906130162,
 'FNR_conformal_copula': 0.04544421722335833,
 'Recall_conformal_copula': 0